# PRCP-1010: Insurance Claim Prediction

**Internship / beginner level project**

### Problem statement
Insurance risk varies from customer to customer. The goal of this project is to build a simple predictive model that helps the insurance team understand which customers are more likely related to a claim / product outcome.

### Tasks covered in this notebook
1. Build a predictive model (`target` prediction)
2. Give suggestions to the insurance marketing team
3. Create a model comparison report
4. Write a short report on challenges faced

### Models used (simple only)
- Logistic Regression
- Decision Tree
- Random Forest

> No complex models (XGBoost / deep learning) are used.

## 1. Import libraries

In [1]:
import os
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
print("Libraries imported successfully")

Libraries imported successfully


## 2. Config / settings

We use a sample of the data so the notebook runs faster on a laptop.
Set `SAMPLE_SIZE = None` if you want to use the full dataset (~595k rows).

In [2]:
DATA_URL = (
    "https://d3ilbtxij3aepc.cloudfront.net/projects/"
    "CDS-Capstone-Projects/PRCP-1010-InsClaimPred.zip"
)
ZIP_PATH = "PRCP-1010-InsClaimPred.zip"
CSV_PATH = os.path.join("data", "Data", "train.csv")

SAMPLE_SIZE = 80000   # change to None for full data
RANDOM_STATE = 42
TEST_SIZE = 0.2

print("Config ready")

Config ready


## 3. Download and load the dataset

The official project dataset is downloaded automatically if it is not already present.

In [3]:
def download_and_extract_data():
    """Download zip if needed and extract train.csv."""
    if os.path.exists(CSV_PATH):
        print(f"[OK] Found dataset at: {CSV_PATH}")
        return

    print("[INFO] Dataset not found. Downloading...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("[OK] Download complete. Extracting...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall("data")

    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError("train.csv was not found after extraction.")
    print(f"[OK] Dataset ready at: {CSV_PATH}")


def load_data():
    """Load CSV and optionally take a stratified sample."""
    download_and_extract_data()
    print("[INFO] Reading CSV (this may take a moment)...")
    df = pd.read_csv(CSV_PATH)
    print(f"[OK] Loaded shape: {df.shape}")

    if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
        # Keep both classes in roughly the same ratio
        df_0 = df[df["target"] == 0]
        df_1 = df[df["target"] == 1]
        n1 = max(1, int(SAMPLE_SIZE * len(df_1) / len(df)))
        n0 = SAMPLE_SIZE - n1
        df = pd.concat(
            [
                df_0.sample(n=n0, random_state=RANDOM_STATE),
                df_1.sample(n=n1, random_state=RANDOM_STATE),
            ],
            ignore_index=True,
        )
        df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f"[OK] Using sample shape: {df.shape}")

    return df


df = load_data()
df.head()

[OK] Found dataset at: data/Data/train.csv
[INFO] Reading CSV (this may take a moment)...


[OK] Loaded shape: (595212, 59)


[OK] Using sample shape: (80000, 59)


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,322068,0,4,1,7,1,0,1,0,0,...,6,3,5,8,0,0,0,1,0,0
1,1232492,0,1,1,7,0,0,0,0,0,...,9,2,2,6,0,0,0,1,1,0
2,269759,0,0,1,5,1,0,1,0,0,...,3,2,0,7,0,0,0,0,0,0
3,1465433,0,1,1,5,1,4,0,0,1,...,5,1,1,6,0,1,0,0,0,0
4,420252,0,2,1,2,1,0,0,0,1,...,2,0,6,2,0,0,0,0,1,1


## 4. Quick data overview

The project document says EDA can be skipped / kept light, so we only check the important points:
- shape
- target class imbalance
- missing values coded as `-1`

In [4]:
print("Rows:", df.shape[0], " Columns:", df.shape[1])
print("\nTarget class counts:")
print(df["target"].value_counts())
print("\nTarget class percentage:")
print((df["target"].value_counts(normalize=True) * 100).round(2))

missing_like = (df == -1).mean().sort_values(ascending=False)
print("\nTop columns where -1 is used as missing value:")
print((missing_like[missing_like > 0].head(10) * 100).round(2))

Rows: 80000  Columns: 59

Target class counts:
target
0    77085
1     2915
Name: count, dtype: int64

Target class percentage:
target
0    96.36
1     3.64
Name: proportion, dtype: float64

Top columns where -1 is used as missing value:
ps_car_03_cat    69.47
ps_car_05_cat    44.91
ps_reg_03        18.21
ps_car_14         7.11
ps_car_07_cat     1.97
ps_ind_05_cat     0.99
ps_car_09_cat     0.10
ps_ind_02_cat     0.03
ps_car_01_cat     0.01
ps_ind_04_cat     0.01
dtype: float64


## 5. Preprocessing (simple intern-level cleaning)

Steps:
1. Drop `id`
2. Replace `-1` with missing values
3. Fill numeric missing values with median
4. Fill categorical/binary missing values with mode
5. One-hot encode small categorical columns

In [5]:
def preprocess(df):
    data = df.copy()
    data = data.drop(columns=["id"])

    target = data["target"]
    features = data.drop(columns=["target"])

    # Column types from naming convention
    cat_cols = [c for c in features.columns if c.endswith("_cat")]
    bin_cols = [c for c in features.columns if c.endswith("_bin")]
    num_cols = [c for c in features.columns if c not in cat_cols + bin_cols]

    # Replace -1 with NaN, then fill simply
    features = features.replace(-1, np.nan)

    for col in num_cols:
        features[col] = features[col].fillna(features[col].median())

    for col in cat_cols + bin_cols:
        mode_vals = features[col].mode(dropna=True)
        fill_value = mode_vals.iloc[0] if len(mode_vals) else 0
        features[col] = features[col].fillna(fill_value)

    # One-hot encode only low-cardinality categories
    small_cat_cols = [c for c in cat_cols if features[c].nunique() <= 20]
    large_cat_cols = [c for c in cat_cols if c not in small_cat_cols]
    features = pd.get_dummies(features, columns=small_cat_cols, drop_first=True)

    print("[OK] Preprocessing done")
    print(f"    Numeric cols: {len(num_cols)}")
    print(f"    Binary cols: {len(bin_cols)}")
    print(f"    One-hot cat cols: {len(small_cat_cols)}")
    print(f"    Left as numeric cat cols: {len(large_cat_cols)}")
    print(f"    Final feature matrix shape: {features.shape}")

    return features, target


X, y = preprocess(df)
X.head()

[OK] Preprocessing done
    Numeric cols: 26
    Binary cols: 17
    One-hot cat cols: 13
    Left as numeric cat cols: 1
    Final feature matrix shape: (80000, 102)


,ps_ind_01,ps_ind_03,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,...,ps_car_06_cat_16,ps_car_06_cat_17,ps_car_07_cat_1.0,ps_car_08_cat_1,ps_car_09_cat_1.0,ps_car_09_cat_2.0,ps_car_09_cat_3.0,ps_car_09_cat_4.0,ps_car_10_cat_1,ps_car_10_cat_2
0,4,7,1,0,0,0,0,0,0,0,...,False,False,True,True,False,False,True,False,True,False
1,1,7,0,0,0,1,0,0,0,0,...,False,False,True,True,False,True,False,False,True,False
2,0,5,1,0,0,0,0,0,0,0,...,False,False,True,True,False,False,False,False,True,False
3,1,5,0,0,1,0,0,0,0,0,...,False,False,True,True,False,True,False,False,True,False
4,2,2,0,0,1,0,0,0,0,0,...,False,False,True,True,False,True,False,False,True,False


## 6. Train / test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target %:")
print((y_train.value_counts(normalize=True) * 100).round(2))

Train shape: (64000, 102)
Test shape: (16000, 102)
Train target %:
target
0    96.36
1     3.64
Name: proportion, dtype: float64


## 7. Helper function to evaluate models

In [7]:
def evaluate_model(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]
    else:
        y_prob = y_pred

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred, zero_division=0),
        "recall": recall_score(y_eval, y_pred, zero_division=0),
        "f1": f1_score(y_eval, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, y_prob),
    }

    print(f"\n--- {name} ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))
    print("Classification Report:")
    print(classification_report(y_eval, y_pred, zero_division=0))
    return metrics

print("Evaluation helper ready")

Evaluation helper ready


## 8. Train simple models

We use `class_weight="balanced"` because the dataset is highly imbalanced.

In [8]:
models = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        True,  # needs scaling
    ),
    "Decision Tree": (
        DecisionTreeClassifier(
            max_depth=6,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        False,
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        False,
    ),
}

results = []
trained = {}

for name, (model, needs_scale) in models.items():
    print(f"\n[INFO] Training {name}...")
    if needs_scale:
        model.fit(X_train_scaled, y_train)
        metrics = evaluate_model(name, model, X_test_scaled, y_test)
    else:
        model.fit(X_train, y_train)
        metrics = evaluate_model(name, model, X_test, y_test)

    results.append(metrics)
    trained[name] = model

print("\nAll models trained")


[INFO] Training Logistic Regression...



--- Logistic Regression ---
Confusion Matrix:
[[9627 5790]
 [ 281  302]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.62      0.76     15417
           1       0.05      0.52      0.09       583

    accuracy                           0.62     16000
   macro avg       0.51      0.57      0.43     16000
weighted avg       0.94      0.62      0.74     16000


[INFO] Training Decision Tree...



--- Decision Tree ---
Confusion Matrix:
[[10039  5378]
 [  326   257]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.65      0.78     15417
           1       0.05      0.44      0.08       583

    accuracy                           0.64     16000
   macro avg       0.51      0.55      0.43     16000
weighted avg       0.93      0.64      0.75     16000


[INFO] Training Random Forest...



--- Random Forest ---
Confusion Matrix:
[[11003  4414]
 [  356   227]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.71      0.82     15417
           1       0.05      0.39      0.09       583

    accuracy                           0.70     16000
   macro avg       0.51      0.55      0.45     16000
weighted avg       0.94      0.70      0.80     16000


All models trained


## 9. Model Comparison Report

We mainly use **ROC-AUC** to choose the best model because accuracy alone can be misleading on imbalanced data.

In [9]:
results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("MODEL COMPARISON REPORT")
display(results_df.round(4))

best_name = results_df.iloc[0]["model"]
print(f"\n[BEST MODEL FOR PRODUCTION] {best_name}")
print(
    "Reason: Best ROC-AUC among simple models. "
    "ROC-AUC is useful for imbalanced claim data."
)

results_df.to_csv("model_comparison_results.csv", index=False)
print("\nSaved: model_comparison_results.csv")

MODEL COMPARISON REPORT


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.6206,0.0496,0.5180,0.0905,0.6081
2,Random Forest,0.7019,0.0489,0.3894,0.0869,0.5868
1,Decision Tree,0.6435,0.0456,0.4408,0.0827,0.5624



[BEST MODEL FOR PRODUCTION] Logistic Regression
Reason: Best ROC-AUC among simple models. ROC-AUC is useful for imbalanced claim data.

Saved: model_comparison_results.csv


## 10. Task 2 — Suggestions to Insurance Marketing Team

We use feature importance / coefficients from the best model to guide simple marketing actions.

In [10]:
model = trained[best_name]

if hasattr(model, "feature_importances_"):
    importance = pd.Series(model.feature_importances_, index=X.columns)
elif hasattr(model, "coef_"):
    importance = pd.Series(np.abs(model.coef_[0]), index=X.columns)
else:
    raise ValueError("No feature importance available for the best model.")

top_features = importance.sort_values(ascending=False).head(10)
print("Top 10 important features:")
display(top_features.round(4).to_frame("importance"))

print(
    """
Practical suggestions:
1) Focus campaigns on customer groups linked to the top features above
   (individual / car / region style signals in this anonymized data).
2) Because very few customers have target=1, do not market randomly.
   Score customers first, then contact high-probability customers.
3) Use a soft offer / education message for medium-score customers
   instead of expensive high-touch sales for everyone.
4) Re-check model scores every few months because customer risk
   and buying behavior can change over time.
5) Combine model score with business rules (age band, vehicle type,
   region, previous claim history) before final outreach.
"""
)

Top 10 important features:


,importance
ps_car_01_cat_7.0,0.1891
ps_car_01_cat_11.0,0.1588
ps_ind_10_bin,0.1513
ps_car_15,0.1419
ps_car_10_cat_2,0.1247
ps_ind_15,0.1210
ps_ind_16_bin,0.1178
ps_ind_05_cat_6.0,0.1176
ps_car_04_cat_4,0.1175
ps_car_01_cat_6.0,0.1041



Practical suggestions:
1) Focus campaigns on customer groups linked to the top features above
   (individual / car / region style signals in this anonymized data).
2) Because very few customers have target=1, do not market randomly.
   Score customers first, then contact high-probability customers.
3) Use a soft offer / education message for medium-score customers
   instead of expensive high-touch sales for everyone.
4) Re-check model scores every few months because customer risk
   and buying behavior can change over time.
5) Combine model score with business rules (age band, vehicle type,
   region, previous claim history) before final outreach.



## 11. Report on Challenges Faced

In [11]:
print(
    """
1) Class imbalance
   - Only about 3-4% of rows are target=1.
   - Technique: used class_weight='balanced' and stratified split.
   - Reason: plain accuracy looks high even if the model misses claims.

2) Missing values coded as -1
   - Dataset uses -1 instead of blank/NaN.
   - Technique: replaced -1 with NaN, then filled median/mode.
   - Reason: models should not treat -1 as a real numeric meaning.

3) Feature names are anonymized
   - Columns like ps_car_*, ps_ind_* do not have business labels.
   - Technique: used feature importance from simple models.
   - Reason: still helps marketing prioritize strongly linked signals.

4) Large dataset size
   - Full data has ~595k rows.
   - Technique: used a stratified sample for faster intern demos.
   - Reason: full-data Random Forest can be slow on a normal laptop.

5) High-cardinality categories
   - Some _cat columns have many unique values.
   - Technique: one-hot encode only small categories; keep large ones numeric.
   - Reason: avoids creating too many columns / memory issues.
"""
)


1) Class imbalance
   - Only about 3-4% of rows are target=1.
   - Technique: used class_weight='balanced' and stratified split.
   - Reason: plain accuracy looks high even if the model misses claims.

2) Missing values coded as -1
   - Dataset uses -1 instead of blank/NaN.
   - Technique: replaced -1 with NaN, then filled median/mode.
   - Reason: models should not treat -1 as a real numeric meaning.

3) Feature names are anonymized
   - Columns like ps_car_*, ps_ind_* do not have business labels.
   - Technique: used feature importance from simple models.
   - Reason: still helps marketing prioritize strongly linked signals.

4) Large dataset size
   - Full data has ~595k rows.
   - Technique: used a stratified sample for faster intern demos.
   - Reason: full-data Random Forest can be slow on a normal laptop.

5) High-cardinality categories
   - Some _cat columns have many unique values.
   - Technique: one-hot encode only small categories; keep large ones numeric.
   - Reason: avo

## 12. Final conclusion

- We completed both project tasks in one simple notebook.
- Beginner-friendly models were compared.
- The best model is selected using ROC-AUC.
- Marketing suggestions were generated from important features.
- Challenges and solutions were documented.

**How to use this notebook**
1. Run all cells from top to bottom
2. Check the model comparison table
3. Read marketing suggestions and challenges report